# Assignment 04 · Thực nghiệm gốc 3: Bóc tách giai thừa $2^3$ và đường cong theo cỡ dữ liệu

| | |
|---|---|
| **Sinh viên** | Nguyễn Duy Nghĩa |
| **Mã sinh viên** | B23DCCN600 |
| **Lớp** | D23CTPM01 |
| **Giảng viên** | PGS.TS Trần Đình Quế |
| **Học phần** | Phát triển các Hệ thống Thông minh |
| **Học kỳ** | Học kỳ 1 năm học 2026 – 2027 |

## Mục tiêu notebook

Báo cáo ở các chương trước hai lần phải thừa nhận rằng mình **không tách được** nguyên nhân, và
notebook này tồn tại để gỡ đúng hai chỗ đó.

**Chỗ thứ nhất, gói "Improved".** Chương 6 so sánh một CNN NumPy *Baseline* với một CNN NumPy
*Improved*, trong đó bản Improved bật **cùng lúc ba** thay đổi: đệm viền (same padding), khởi tạo
He Normal, và lịch giảm tốc độ học. Vì ba thay đổi được bật một lượt nên bảng kết quả chỉ nói được
"gói cải tiến cộng thêm bao nhiêu điểm", chứ tuyệt nhiên không nói được **thay đổi nào** tạo ra
phần cộng thêm đó. Câu hỏi càng đáng đặt ra khi ở một miền khác của cùng báo cáo, gói Improved
thậm chí **không** vượt được Baseline. Nếu một trong ba yếu tố thực ra vô dụng hoặc có hại, gói
gộp sẽ che mất điều đó.

Notebook chạy đủ **tám tổ hợp bật/tắt** của ba yếu tố, tức thiết kế giai thừa đầy đủ $2^3$, rồi
tính **hiệu ứng chính** của từng yếu tố và cả **ba hiệu ứng tương tác bậc hai**. Lý do phải làm
giai thừa đầy đủ thay vì thử từng yếu tố một: hai yếu tố có thể **chỉ có ích khi đi cùng nhau**,
và phép thử lần lượt từng cái sẽ bỏ sót đúng trường hợp đó.

**Chỗ thứ hai, khoảng cách NumPy với framework.** Chương 6 và Chương 7 nói thẳng rằng chênh lệch
giữa mô hình NumPy thuần và mô hình framework **trộn lẫn ba nguyên nhân**: ít dữ liệu huấn luyện
hơn, kiến trúc nông hơn, và tầng hiện thực khác nhau. Lời thừa nhận đó trung thực nhưng chưa đủ.
Notebook đo trực tiếp **thành phần do dữ liệu** bằng cách huấn luyện cùng một kiến trúc framework
trên dãy cỡ dữ liệu tăng dần, rồi đọc giá trị tại đúng cỡ mà mô hình NumPy đã dùng.

Cần nói trước một giới hạn của thiết kế, để phần kết luận không vượt quá dữ liệu: thí nghiệm này
tách được **hai** phần (phần do dữ liệu, và phần còn lại), chứ không tách được **ba** phần. Phần
còn lại vẫn gộp chung kiến trúc với tầng hiện thực, nên nó là **cận trên** của ảnh hưởng kiến
trúc chứ không phải phép đo sạch của riêng kiến trúc.

## Đầu ra

- `analysis/reports/metrics_ablation.json`
- `analysis/reports/figures/fig_ablation_factorial.png`
- `analysis/reports/figures/fig_learning_curve.png`
- `analysis/reports/figures/fig_gap_decomposition.png`

## 1. Nhập thư viện, cấu hình hạt giống và thiết bị

Theo Mục 1 của hợp đồng tích hợp, mọi notebook PyTorch phải chạy trên GPU và phải in ra tên thiết
bị. Notebook này huấn luyện **72 mô hình** (48 mô hình cho phần giai thừa và 24 mô hình cho đường
cong theo cỡ dữ liệu), nên GPU không phải tiện nghi mà là điều kiện cần để nằm trong ngân sách
thời gian.

In [1]:
import os, sys, json, time, math, copy, itertools, datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert torch.cuda.is_available(), 'Phai chay tren GPU; kiem tra lai venv'
torch.backends.cudnn.benchmark = True      # cho phep cuDNN tu do kich thuoc kernel toi uu

plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# Duong dan tuong doi tinh tu thu muc analysis/notebooks/
DATA_PATH     = '../../mnist/data/mnist.npz'
MNIST_METRICS = '../../mnist/reports/metrics_mnist.json'
MNIST_MODELS  = os.path.abspath('../../mnist/models')
FIG_DIR       = '../reports/figures'
REP_DIR       = '../reports'
os.makedirs(FIG_DIR, exist_ok=True)

# Nap lai dinh nghia kien truc framework cua Chuong 7 thay vi cai lai,
# de bao dam duong cong theo co du lieu dung DUNG mo hinh da bao cao o do.
sys.dont_write_bytecode = True             # khong ghi __pycache__ sang thu muc mien khac
sys.path.insert(0, MNIST_MODELS)
from mnist_cnn_def import MnistCNN

print('NumPy      ', np.__version__)
print('PyTorch    ', torch.__version__)
print('Thiet bi   ', DEVICE, '|', torch.cuda.get_device_name(0))
print('VRAM tong  %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1024**3))
print('Kien truc framework nap tu:', os.path.join(MNIST_MODELS, 'mnist_cnn_def.py'))

NumPy       2.5.1
PyTorch     2.13.0+cu126
Thiet bi    cuda | NVIDIA GeForce RTX 4060 Laptop GPU
VRAM tong  8.0 GB
Kien truc framework nap tu: D:\Python\Intelligent-System-Development\src\Assignment 04\mnist\models\mnist_cnn_def.py


## 2. Nạp dữ liệu và tái lập đúng phép chia của Chương 6 và Chương 7

Toàn bộ giá trị của notebook này nằm ở chỗ nó **so sánh được** với hai chương trước. Muốn vậy thì
phép chia train/validation, hằng số chuẩn hóa, và tập con 12 000 ảnh của mô hình NumPy phải được
tái lập **y hệt**, chứ không phải dựng lại một cách gần đúng.

Ba biện pháp bảo đảm điều đó:

1. Dùng lại chính lệnh `train_test_split(test_size=0.2, stratify=y, random_state=42)` của hợp đồng,
   cho ra 48 000 ảnh train và 12 000 ảnh validation.
2. Đọc hằng số chuẩn hóa từ `mnist/models/mnist_preproc.json` thay vì tính lại, rồi **đối chiếu**
   với giá trị tính lại từ nhánh train để chắc chắn hai bên khớp nhau.
3. Đọc cỡ tập con NumPy từ `mnist/reports/metrics_mnist.json` thay vì viết cứng con số 12 000, và
   dựng tập con bằng đúng lệnh mà Chương 6 đã dùng.

Chuẩn hóa theo công thức của hợp đồng: $x_{\text{norm}} = \dfrac{x_{\text{uint8}}/255 - \mu}{\sigma}$,
với $\mu$ và $\sigma$ học từ 48 000 ảnh nhánh train. Tập test 10 000 ảnh gốc **không bao giờ** tham
gia chọn epoch, chỉ dùng để báo cáo con số cuối cùng.

**Một lưu ý về tính đồng thời.** Tệp `metrics_mnist.json` đang được một quy trình khác ghi lại
trong lúc notebook này chạy, vì các chương trước được huấn luyện lại trên GPU. Notebook vì thế đọc
tệp qua hàm `load_mnist_ref()` có cơ chế thử lại và kiểm tra tính đầy đủ, để không bao giờ đọc phải
một tệp đang viết dở. Quan trọng hơn, những con số đi thẳng vào phép bóc tách được **đọc lại lần
nữa ở ngay trước chỗ dùng** (Mục 4.4), chứ không dùng bản đã đọc từ đầu notebook.

In [2]:
_d = np.load(DATA_PATH)
x_train_raw, y_train_raw = _d['x_train'], _d['y_train'].astype(np.int64)
x_test_raw,  y_test_raw  = _d['x_test'],  _d['y_test'].astype(np.int64)

# --- Chia train / validation co phan tang, giong het hop dong ---
idx_all = np.arange(len(x_train_raw))
idx_tr, idx_va = train_test_split(idx_all, test_size=0.2,
                                  stratify=y_train_raw, random_state=RANDOM_SEED)
x_tr_u8, y_tr_full = x_train_raw[idx_tr], y_train_raw[idx_tr]
x_va_u8, y_va_full = x_train_raw[idx_va], y_train_raw[idx_va]

# --- Hang so chuan hoa: doc lai tu Chuong 7 roi doi chieu voi ban tinh lai ---
with open(os.path.join(MNIST_MODELS, 'mnist_preproc.json'), encoding='utf-8') as f:
    _pp = json.load(f)
MEAN, STD = float(_pp['mean']), float(_pp['std'])
_mean_check = float((x_tr_u8.astype(np.float32) / 255.0).mean())
_std_check  = float((x_tr_u8.astype(np.float32) / 255.0).std())
assert abs(MEAN - _mean_check) < 1e-6 and abs(STD - _std_check) < 1e-6, \
    'Hang so chuan hoa khong khop voi nhanh train; phep chia da lech'
print(f'MEAN = {MEAN:.10f}  (tinh lai {_mean_check:.10f})')
print(f'STD  = {STD:.10f}  (tinh lai {_std_check:.10f})   -> khop')


def to_gpu(x_u8):
    '''uint8 (N,28,28) -> tensor float32 (N,1,28,28) da chuan hoa, nam san tren GPU.'''
    x = x_u8.astype(np.float32) / 255.0
    x = (x - MEAN) / STD
    return torch.from_numpy(x[:, None, :, :]).to(DEVICE)


X_TR_FULL = to_gpu(x_tr_u8);    Y_TR_FULL = torch.from_numpy(y_tr_full).to(DEVICE)
X_VA_FULL = to_gpu(x_va_u8);    Y_VA_FULL = torch.from_numpy(y_va_full).to(DEVICE)
X_TE      = to_gpu(x_test_raw); Y_TE      = torch.from_numpy(y_test_raw).to(DEVICE)


def load_mnist_ref(retries=30, wait=10.0, verbose=True):
    '''Doc metrics_mnist.json mot cach an toan truoc kha nang tep dang bi ghi lai.

    Tra ve (doi tuong JSON, chuoi thoi diem sua tep). Chi chap nhan ban doc khi du ca bon
    khoa model, nen khong bao gio dung phai mot tep viet do.
    '''
    last = None
    for _ in range(retries):
        try:
            mtime = datetime.datetime.fromtimestamp(
                os.path.getmtime(MNIST_METRICS)).strftime('%Y-%m-%d %H:%M:%S')
            with open(MNIST_METRICS, encoding='utf-8') as fh:
                obj = json.load(fh)
            need = ('numpy_baseline', 'numpy_improved', 'pytorch', 'tensorflow')
            models = obj.get('models', {})
            miss = [k for k in need if k not in models or 'accuracy' not in models[k]]
            if not miss:
                if verbose:
                    print(f'Doc {MNIST_METRICS} (sua luc {mtime}) -> du ca 4 mo hinh')
                return obj, mtime
            last = f'thieu {miss}'
        except (json.JSONDecodeError, OSError) as e:
            last = repr(e)
        if verbose:
            print(f'  tep dang duoc ghi lai ({last}), doi {wait:.0f}s roi thu lai...')
        time.sleep(wait)
    raise RuntimeError(f'Khong doc duoc {MNIST_METRICS}: {last}')


MNIST_REF, MNIST_REF_MTIME = load_mnist_ref()


def _dig(obj, key, default=None):
    '''Tim khoa o cap cao nhat hoac long trong "dataset", theo dung ghi chu cua hop dong.'''
    if key in obj:
        return obj[key]
    if 'dataset' in obj and key in obj['dataset']:
        return obj['dataset'][key]
    return default


_sub = _dig(MNIST_REF, 'numpy_subset', {})
N_SMALL   = int(_sub.get('n_train', 12000))
N_SUB_VAL = int(_sub.get('n_val', 3000))

# Tap con NumPy, dung dung lenh cua Chuong 6 nen tai lap chinh xac tung anh
sub_tr, _ = train_test_split(np.arange(len(y_tr_full)), train_size=N_SMALL,
                             stratify=y_tr_full, random_state=RANDOM_SEED)
sub_va, _ = train_test_split(np.arange(len(y_va_full)), train_size=N_SUB_VAL,
                             stratify=y_va_full, random_state=RANDOM_SEED)
X_SUB_TR = X_TR_FULL[torch.from_numpy(sub_tr).to(DEVICE)]
Y_SUB_TR = Y_TR_FULL[torch.from_numpy(sub_tr).to(DEVICE)]
X_SUB_VA = X_VA_FULL[torch.from_numpy(sub_va).to(DEVICE)]
Y_SUB_VA = Y_VA_FULL[torch.from_numpy(sub_va).to(DEVICE)]

print()
print(f'Train day du   : {tuple(X_TR_FULL.shape)}')
print(f'Validation     : {tuple(X_VA_FULL.shape)}')
print(f'Test           : {tuple(X_TE.shape)}')
print(f'Tap con NumPy  : train {tuple(X_SUB_TR.shape)}, val {tuple(X_SUB_VA.shape)} '
      f'(doc tu metrics_mnist.json: n_train={N_SMALL}, n_val={N_SUB_VAL})')
print(f'Phan phoi lop tap con: {np.bincount(y_tr_full[sub_tr], minlength=10).tolist()}')
print(f'Bo nho GPU dang dung : {torch.cuda.memory_allocated()/1024**2:.0f} MB')

MEAN = 0.1307886839  (tinh lai 0.1307886839)
STD  = 0.3082403541  (tinh lai 0.3082403541)   -> khop


Doc ../../mnist/reports/metrics_mnist.json (sua luc 2026-09-16 12:27:11) -> du ca 4 mo hinh

Train day du   : (48000, 1, 28, 28)
Validation     : (12000, 1, 28, 28)
Test           : (10000, 1, 28, 28)
Tap con NumPy  : train (12000, 1, 28, 28), val (3000, 1, 28, 28) (doc tu metrics_mnist.json: n_train=12000, n_val=3000)
Phan phoi lop tap con: [1184, 1349, 1192, 1226, 1168, 1084, 1184, 1253, 1170, 1190]
Bo nho GPU dang dung : 256 MB


## 3. Phần A: thiết kế giai thừa đầy đủ $2^3$ cho gói "Improved"

### 3.1 Cơ sở của thiết kế giai thừa

Gọi ba yếu tố là $A$ (đệm viền), $B$ (khởi tạo He Normal), $C$ (lịch giảm tốc độ học), mỗi yếu tố
nhận hai mức: tắt $(-)$ và bật $(+)$. Thiết kế giai thừa đầy đủ chạy toàn bộ $2^3 = 8$ tổ hợp. Ký
hiệu $\bar{y}_{A^+}$ là accuracy trung bình của bốn tổ hợp có $A$ bật, **hiệu ứng chính** của $A$ là

$$ E_A \;=\; \bar{y}_{A^+} - \bar{y}_{A^-} \;=\; \frac{1}{4}\sum_{b,c \in \{-,+\}} \big( y_{+bc} - y_{-bc} \big) $$

nghĩa là trung bình của bốn phép so sánh bật/tắt, mỗi phép thực hiện trong một bối cảnh khác nhau
của hai yếu tố còn lại. Đây chính là điểm mạnh của thiết kế giai thừa so với cách thử từng yếu tố
một: hiệu ứng chính được ước lượng trên **toàn bộ** tám điểm chứ không phải hai điểm, nên với cùng
một số lần chạy, phương sai của ước lượng nhỏ hơn hẳn.

**Hiệu ứng tương tác** bậc hai giữa $A$ và $B$ được định nghĩa là nửa hiệu của hai hiệu ứng đơn:

$$ E_{AB} \;=\; \tfrac{1}{2}\Big[ \big( \bar{y}_{A^+B^+} - \bar{y}_{A^-B^+} \big) - \big( \bar{y}_{A^+B^-} - \bar{y}_{A^-B^-} \big) \Big] $$

Diễn giải bằng lời: $E_{AB}$ đo xem **ích lợi của $A$ có thay đổi hay không khi $B$ được bật**. Nếu
$E_{AB}$ xấp xỉ 0 thì hai yếu tố cộng tính, và lúc đó phép thử từng cái một sẽ cho cùng kết luận.
Nếu $E_{AB}$ lớn thì hai yếu tố phụ thuộc lẫn nhau, và **chỉ** thiết kế giai thừa mới phát hiện ra.
Đây không phải khả năng lý thuyết suông: He Normal được thiết kế để giữ phương sai tín hiệu qua các
tầng, còn đệm viền làm thay đổi số chiều đầu vào của tầng Dense từ 400 lên 784, tức thay đổi chính
cái `fan_in` mà công thức He dùng, nên có lý do vật lý để chờ đợi một tương tác giữa $A$ và $B$.

### 3.2 Một yếu tố gây nhiễu thứ tư mà nhãn gói che mất

Khi đọc lại mã nguồn Chương 6 để dựng lại kiến trúc, báo cáo phát hiện gói "Improved" thực ra
không đổi ba thứ mà **bốn**: ngoài đệm viền, He Normal và lịch giảm tốc độ học, nó còn nâng tốc độ
học nền từ $10^{-3}$ lên $2\times 10^{-3}$. Chi tiết này không xuất hiện trong nhãn "Improved" và
vì thế rất dễ bị bỏ qua.

Để tốc độ học nền không trộn vào kết quả, notebook **giữ cố định** giá trị này trong toàn bộ tám tổ
hợp, nên yếu tố $C$ ở đây đo đúng một việc là **có nên hạ dần tốc độ học theo thời gian hay không**,
chứ không đo việc đổi tốc độ học nền. Vòng chạy chính dùng $\eta_0 = 2\times10^{-3}$, đúng giá trị
của bản Improved, nên tổ hợp bật cả ba yếu tố tái lập đúng cấu hình đã báo cáo ở Chương 6. Vì lựa
chọn $\eta_0$ vẫn có thể ảnh hưởng tới kết luận về yếu tố $C$, Mục 3.6 chạy lại toàn bộ thiết kế ở
$\eta_0 = 10^{-3}$, đúng giá trị của bản Baseline, như một phép kiểm tra tính bền vững.

### 3.3 Giao thức thực nghiệm

| Hạng mục | Giá trị | Lý do |
|---|---|---|
| Kiến trúc | Bản sao PyTorch của CNN NumPy Chương 6 | Kết luận về ba yếu tố chuyển giao được |
| Tập huấn luyện | Đúng tập con 12 000 ảnh của Chương 6 | Cố định cho cả tám tổ hợp |
| Tập validation | Đúng tập con 3 000 ảnh của Chương 6 | Dùng để chọn epoch, không dùng test |
| Tập kiểm thử | 10 000 ảnh gốc | Chỉ chạm một lần cho mỗi mô hình |
| Tối ưu | Adam, 16 epoch, batch 128 | Xem ghi chú sai lệch bên dưới |
| Hạt giống | 42, 43, 44 cho mỗi tổ hợp | Cho phép đo nhiễu khởi tạo |
| Lịch giảm LR (khi bật) | $\eta_0 \cdot 0{,}5^{\lfloor (e-1)/4 \rfloor}$ | Đúng lịch của Chương 6 |

**Ghi chú sai lệch so với Chương 6.** Chương 6 dùng batch 64, notebook này dùng batch 128. Lý do
thuần túy là ngân sách tính toán: ở batch 64 mô hình quá nhỏ nên thời gian bị chi phí gọi nhân GPU
chi phối, đo được 4,13 giây mỗi epoch so với 1,13 giây ở batch 128, tức chậm hơn ba lần rưỡi mà
không tăng lượng tính toán thực. Với 48 lần chạy thì khác biệt này là vài chục phút. Batch 128 được
giữ **nguyên cho cả tám tổ hợp**, nên nó không thể làm lệch phép so sánh giữa các tổ hợp, vốn là
đại lượng mà notebook quan tâm. Hệ quả duy nhất là accuracy tuyệt đối sẽ thấp hơn Chương 6 một
chút, vì cùng số epoch thì batch lớn hơn nghĩa là ít bước cập nhật hơn.

Mỗi lần chạy chọn **checkpoint có accuracy validation cao nhất** rồi mới đo trên tập test, giống
cách Chương 6 chọn `best_epoch`. Cách này bảo vệ công bằng cho các tổ hợp hội tụ nhanh hoặc chậm
khác nhau, vì tổ hợp khởi tạo kém sẽ không bị phạt oan chỉ do nó cần nhiều epoch hơn.

In [3]:
class SmallCNN(nn.Module):
    '''Ban sao PyTorch cua CNN 2D thuan NumPy o Chuong 6 (muc 4 hop dong, bien the MNIST).

    Conv2D(8,K=3) -> ReLU -> MaxPool(2) -> Conv2D(16,K=3) -> ReLU -> MaxPool(2)
                  -> Flatten -> Dense(64) -> ReLU -> Dense(10)

    Hai yeu to duoc tham so hoa de chay thiet ke giai thua:
      padding : 0 = khong dem vien (Baseline), 1 = dem vien same (Improved)
      he_init : False = randn * 0.01 (Baseline), True = He Normal (Improved)
    '''

    def __init__(self, padding: int = 1, he_init: bool = True, seed: int = RANDOM_SEED):
        super().__init__()
        self.padding = int(padding)
        # 28 -> 28 -> 14 -> 14 -> 7 khi padding=1 ; 28 -> 26 -> 13 -> 11 -> 5 khi padding=0
        side = 7 if self.padding == 1 else 5
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=self.padding)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=self.padding)
        self.fc1   = nn.Linear(16 * side * side, 64)
        self.fc2   = nn.Linear(64, 10)
        self.pool, self.relu, self.flat = nn.MaxPool2d(2), nn.ReLU(), nn.Flatten()
        self.side, self.he_init = side, bool(he_init)
        self._init_weights(seed)

    def _init_weights(self, seed: int):
        '''He Normal: sigma = sqrt(2 / fan_in), voi fan_in = C_in*K*K o Conv va so dau vao o Dense.
        Bien the Baseline dung sigma = 0.01 co dinh, khong phu thuoc fan_in.'''
        g = torch.Generator().manual_seed(seed)
        self.sigmas = {}
        for name, m in (('conv1', self.conv1), ('conv2', self.conv2),
                        ('fc1', self.fc1), ('fc2', self.fc2)):
            fan_in = m.weight[0].numel()
            sigma = math.sqrt(2.0 / fan_in) if self.he_init else 0.01
            with torch.no_grad():
                m.weight.copy_(torch.randn(m.weight.shape, generator=g) * sigma)
                m.bias.zero_()
            self.sigmas[name] = float(sigma)

    def forward(self, x):
        h = self.pool(self.relu(self.conv1(x)))
        h = self.pool(self.relu(self.conv2(h)))
        h = self.relu(self.fc1(self.flat(h)))
        return self.fc2(h)


# --- Doi chieu so tham so voi Chuong 6 de chung minh day dung la ban sao ---
n_par = lambda m: sum(p.numel() for p in m.parameters())
p_base = n_par(SmallCNN(padding=0, he_init=False))
p_impr = n_par(SmallCNN(padding=1, he_init=True))
ref_base = MNIST_REF['models']['numpy_baseline']['params']
ref_impr = MNIST_REF['models']['numpy_improved']['params']

print(f'padding=0 : {p_base:>7,} tham so | Chuong 6 numpy_baseline : {ref_base:>7,}')
print(f'padding=1 : {p_impr:>7,} tham so | Chuong 6 numpy_improved : {ref_impr:>7,}')
assert p_base == ref_base and p_impr == ref_impr, 'So tham so lech, kien truc chua phai ban sao'
print('-> Trung khop tung tham so, ban sao PyTorch dung bang kien truc NumPy cua Chuong 6.')

_m = SmallCNN(padding=1, he_init=True)
print()
print('Sigma He Normal:', {k: round(v, 6) for k, v in _m.sigmas.items()})
print('  doi chieu sqrt(2/9)   = %.6f' % math.sqrt(2 / 9))
print('  doi chieu sqrt(2/784) = %.6f' % math.sqrt(2 / 784))

padding=0 :  27,562 tham so | Chuong 6 numpy_baseline :  27,562
padding=1 :  52,138 tham so | Chuong 6 numpy_improved :  52,138
-> Trung khop tung tham so, ban sao PyTorch dung bang kien truc NumPy cua Chuong 6.

Sigma He Normal: {'conv1': 0.471405, 'conv2': 0.166667, 'fc1': 0.050508, 'fc2': 0.176777}
  doi chieu sqrt(2/9)   = 0.471405
  doi chieu sqrt(2/784) = 0.050508


In [4]:
@torch.no_grad()
def evaluate(model, X, y, bs=2048):
    '''Danh gia theo lo. Tra ve (loss trung binh, accuracy).'''
    model.eval()
    loss_sum, correct = 0.0, 0
    for s in range(0, len(X), bs):
        out = model(X[s:s + bs])
        loss_sum += float(F.cross_entropy(out, y[s:s + bs], reduction='sum'))
        correct  += int((out.argmax(1) == y[s:s + bs]).sum())
    return loss_sum / len(X), correct / len(X)


def train_epochs(model, Xtr, ytr, Xva, yva, *, epochs, bs, lr0,
                 lr_decay=None, decay_every=4, seed=RANDOM_SEED):
    '''Huan luyen theo so epoch co dinh, giu lai checkpoint tot nhat tren validation.

    lr_decay=None nghia la toc do hoc co dinh; nguoc lai dung
    lr_e = lr0 * lr_decay ** floor((e-1)/decay_every), dung lich cua Chuong 6.
    '''
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr0, fused=True)
    g = torch.Generator(device=DEVICE); g.manual_seed(seed)
    n = len(Xtr)
    hist = {'val_acc': [], 'lr': []}
    best = {'acc': -1.0, 'epoch': 0, 'state': None}

    torch.cuda.synchronize(); t0 = time.time()
    for ep in range(1, epochs + 1):
        lr_ep = lr0 if lr_decay is None else lr0 * (lr_decay ** ((ep - 1) // decay_every))
        for pg in opt.param_groups:
            pg['lr'] = lr_ep
        model.train()
        perm = torch.randperm(n, generator=g, device=DEVICE)
        Xs, ys = Xtr[perm], ytr[perm]
        for s in range(0, n, bs):
            opt.zero_grad(set_to_none=True)
            F.cross_entropy(model(Xs[s:s + bs]), ys[s:s + bs]).backward()
            opt.step()
        _, va = evaluate(model, Xva, yva)
        hist['val_acc'].append(va); hist['lr'].append(lr_ep)
        if va > best['acc']:
            best = {'acc': va, 'epoch': ep, 'state': copy.deepcopy(model.state_dict())}
    torch.cuda.synchronize()
    return best, hist, time.time() - t0


def train_steps(model, Xtr, ytr, Xva, yva, *, total_steps, bs, lr0, eval_every, seed):
    '''Huan luyen theo NGAN SACH BUOC CAP NHAT co dinh, khong phai so epoch co dinh.

    Dung cho duong cong theo co du lieu: neu co dinh so epoch thi diem n nho se nhan
    it buoc cap nhat hon han, va duong cong se tron lan "it du lieu" voi "it huan luyen".
    '''
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr0, fused=True)
    g = torch.Generator(device=DEVICE); g.manual_seed(seed)
    n, step = len(Xtr), 0
    best, curve = {'acc': -1.0, 'step': 0, 'state': None}, []

    torch.cuda.synchronize(); t0 = time.time()
    while step < total_steps:
        perm = torch.randperm(n, generator=g, device=DEVICE)
        Xs, ys = Xtr[perm], ytr[perm]
        for s in range(0, n, bs):
            if step >= total_steps:
                break
            model.train()
            opt.zero_grad(set_to_none=True)
            F.cross_entropy(model(Xs[s:s + bs]), ys[s:s + bs]).backward()
            opt.step()
            step += 1
            if step % eval_every == 0 or step == total_steps:
                _, va = evaluate(model, Xva, yva)
                curve.append((step, va))
                if va > best['acc']:
                    best = {'acc': va, 'step': step, 'state': copy.deepcopy(model.state_dict())}
    torch.cuda.synchronize()
    return best, curve, time.time() - t0


print('Da dinh nghia evaluate / train_epochs / train_steps.')
print('train_epochs -> dung cho Phan A (n co dinh, so epoch co dinh)')
print('train_steps  -> dung cho Phan B (n thay doi, ngan sach buoc cap nhat co dinh)')

Da dinh nghia evaluate / train_epochs / train_steps.
train_epochs -> dung cho Phan A (n co dinh, so epoch co dinh)
train_steps  -> dung cho Phan B (n thay doi, ngan sach buoc cap nhat co dinh)


In [5]:
FACT_EPOCHS  = 16
FACT_BATCH   = 128
FACT_SEEDS   = [42, 43, 44]
LR_PRIMARY   = 2e-3        # bang toc do hoc nen cua ban Improved o Chuong 6
LR_ROBUST    = 1e-3        # bang toc do hoc nen cua ban Baseline o Chuong 6
DECAY_GAMMA, DECAY_EVERY = 0.5, 4


def run_factorial(lr0, seeds=FACT_SEEDS, epochs=FACT_EPOCHS, verbose=True):
    '''Chay du 8 to hop 2^3, moi to hop lap lai voi tung hat giong. Tra ve danh sach 8 ban ghi.'''
    rows = []
    for pad, he, dec in itertools.product((0, 1), (0, 1), (0, 1)):
        accs, vals, eps, secs = [], [], [], []
        for sd in seeds:
            model = SmallCNN(padding=pad, he_init=bool(he), seed=sd)
            best, _, el = train_epochs(
                model, X_SUB_TR, Y_SUB_TR, X_SUB_VA, Y_SUB_VA,
                epochs=epochs, bs=FACT_BATCH, lr0=lr0,
                lr_decay=(DECAY_GAMMA if dec else None), decay_every=DECAY_EVERY, seed=sd)
            model.load_state_dict(best['state'])
            _, te = evaluate(model, X_TE, Y_TE)
            accs.append(te); vals.append(best['acc']); eps.append(best['epoch']); secs.append(el)
        row = {'padding': bool(pad), 'he_init': bool(he), 'lr_decay': bool(dec),
               'accuracy': float(np.mean(accs)), 'accuracy_std': float(np.std(accs, ddof=0)),
               'accuracy_seeds': [float(a) for a in accs],
               'val_accuracy': float(np.mean(vals)), 'best_epoch_mean': float(np.mean(eps)),
               'params': int(sum(p.numel() for p in SmallCNN(padding=pad).parameters())),
               'train_time_s': float(np.sum(secs))}
        rows.append(row)
        if verbose:
            print(f'  [{int(pad)}{int(he)}{int(dec)}] dem={int(pad)} He={int(he)} '
                  f'giamLR={int(dec)} | '
                  f'test = {row["accuracy"]*100:.2f} +/- {row["accuracy_std"]*100:.2f} % | '
                  f'val = {row["val_accuracy"]*100:.2f} % | '
                  f'epoch tot nhat ~ {row["best_epoch_mean"]:.1f} | {row["train_time_s"]:.0f}s')
    return rows


print(f'Vong chay chinh: lr0 = {LR_PRIMARY}, {len(FACT_SEEDS)} hat giong x 8 to hop = '
      f'{len(FACT_SEEDS)*8} lan huan luyen, {FACT_EPOCHS} epoch moi lan')
print(f'Tap huan luyen {len(X_SUB_TR)} anh, validation {len(X_SUB_VA)} anh, test {len(X_TE)} anh')
print('Ma ba chu so theo thu tu [dem vien][He Normal][giam LR]')
print('-' * 104)
t_fact = time.time()
FACT_ROWS = run_factorial(LR_PRIMARY)
print('-' * 104)
print(f'Tong thoi gian phan giai thua chinh: {time.time()-t_fact:.0f} giay')

Vong chay chinh: lr0 = 0.002, 3 hat giong x 8 to hop = 24 lan huan luyen, 16 epoch moi lan
Tap huan luyen 12000 anh, validation 3000 anh, test 10000 anh
Ma ba chu so theo thu tu [dem vien][He Normal][giam LR]
--------------------------------------------------------------------------------------------------------


  [000] dem=0 He=0 giamLR=0 | test = 97.47 +/- 0.13 % | val = 97.71 % | epoch tot nhat ~ 15.7 | 64s


  [001] dem=0 He=0 giamLR=1 | test = 97.17 +/- 0.20 % | val = 97.26 % | epoch tot nhat ~ 13.0 | 71s


  [010] dem=0 He=1 giamLR=0 | test = 97.89 +/- 0.02 % | val = 97.80 % | epoch tot nhat ~ 13.0 | 63s


  [011] dem=0 He=1 giamLR=1 | test = 97.71 +/- 0.09 % | val = 97.79 % | epoch tot nhat ~ 10.7 | 69s


  [100] dem=1 He=0 giamLR=0 | test = 97.47 +/- 0.17 % | val = 97.62 % | epoch tot nhat ~ 14.3 | 61s


  [101] dem=1 He=0 giamLR=1 | test = 97.20 +/- 0.47 % | val = 97.19 % | epoch tot nhat ~ 14.3 | 67s


  [110] dem=1 He=1 giamLR=0 | test = 97.87 +/- 0.11 % | val = 98.13 % | epoch tot nhat ~ 14.3 | 73s


  [111] dem=1 He=1 giamLR=1 | test = 97.80 +/- 0.12 % | val = 98.03 % | epoch tot nhat ~ 13.0 | 82s
--------------------------------------------------------------------------------------------------------
Tong thoi gian phan giai thua chinh: 563 giay


In [6]:
def factorial_frame(rows):
    '''Bang thiet ke giai thua theo quy uoc dau: "-" la tat, "+" la bat.'''
    sgn = lambda b: '+' if b else '-'
    df = pd.DataFrame([{
        'Đệm viền': sgn(r['padding']),
        'He Normal': sgn(r['he_init']),
        'Giảm LR': sgn(r['lr_decay']),
        'Tham số': r['params'],
        'Val (%)': round(r['val_accuracy'] * 100, 2),
        'Test (%)': round(r['accuracy'] * 100, 2),
        'Độ lệch chuẩn (%)': round(r['accuracy_std'] * 100, 3),
        'Ba hạt giống (%)': ', '.join(f'{a*100:.2f}' for a in r['accuracy_seeds']),
    } for r in rows])
    df.index = [f'{i+1}' for i in range(len(df))]
    return df


DF_FACT = factorial_frame(FACT_ROWS)
best_i  = int(np.argmax([r['accuracy'] for r in FACT_ROWS]))
worst_i = int(np.argmin([r['accuracy'] for r in FACT_ROWS]))
bundle  = next(i for i, r in enumerate(FACT_ROWS)
               if r['padding'] and r['he_init'] and r['lr_decay'])
plain   = next(i for i, r in enumerate(FACT_ROWS)
               if not r['padding'] and not r['he_init'] and not r['lr_decay'])

print('Bang thiet ke giai thua 2^3 (trung binh 3 hat giong, accuracy tren 10 000 anh test)')
print()
print(DF_FACT.to_string())
print()
print(f'Tổ hợp tốt nhất : hàng {best_i+1}  = {FACT_ROWS[best_i]["accuracy"]*100:.2f} %')
print(f'Tổ hợp kém nhất : hàng {worst_i+1} = {FACT_ROWS[worst_i]["accuracy"]*100:.2f} %')
print(f'Gói Improved (bật cả ba) : hàng {bundle+1} = {FACT_ROWS[bundle]["accuracy"]*100:.2f} %')
print(f'Gói Baseline (tắt cả ba) : hàng {plain+1}  = {FACT_ROWS[plain]["accuracy"]*100:.2f} %')
print(f'Chênh lệch gói Improved so với Baseline : '
      f'{(FACT_ROWS[bundle]["accuracy"]-FACT_ROWS[plain]["accuracy"])*100:+.2f} điểm phần trăm')
print(f'Biên độ toàn bảng (tốt nhất trừ kém nhất) : '
      f'{(FACT_ROWS[best_i]["accuracy"]-FACT_ROWS[worst_i]["accuracy"])*100:.2f} điểm phần trăm')
DF_FACT

Bang thiet ke giai thua 2^3 (trung binh 3 hat giong, accuracy tren 10 000 anh test)



  Đệm viền He Normal Giảm LR  Tham số  Val (%)  Test (%)  Độ lệch chuẩn (%)     Ba hạt giống (%)
1        -         -       -    27562    97.71     97.47              0.129  97.64, 97.33, 97.43
2        -         -       +    27562    97.26     97.17              0.198  97.45, 97.03, 97.03
3        -         +       -    27562    97.80     97.89              0.022  97.92, 97.87, 97.88
4        -         +       +    27562    97.79     97.71              0.094  97.72, 97.59, 97.82
5        +         -       -    52138    97.62     97.47              0.165  97.32, 97.70, 97.39
6        +         -       +    52138    97.19     97.20              0.474  96.76, 97.86, 96.99
7        +         +       -    52138    98.13     97.87              0.108  97.77, 98.02, 97.82
8        +         +       +    52138    98.03     97.80              0.118  97.63, 97.88, 97.88

Tổ hợp tốt nhất : hàng 3  = 97.89 %
Tổ hợp kém nhất : hàng 2 = 97.17 %
Gói Improved (bật cả ba) : hàng 8 = 97.80 %
Gói Baselin

,Đệm viền,He Normal,Giảm LR,Tham số,Val (%),Test (%),Độ lệch chuẩn (%),Ba hạt giống (%)
1,-,-,-,27562,97.71,97.47,0.129,"97.64, 97.33, 97.43"
2,-,-,+,27562,97.26,97.17,0.198,"97.45, 97.03, 97.03"
3,-,+,-,27562,97.80,97.89,0.022,"97.92, 97.87, 97.88"
4,-,+,+,27562,97.79,97.71,0.094,"97.72, 97.59, 97.82"
5,+,-,-,52138,97.62,97.47,0.165,"97.32, 97.70, 97.39"
6,+,-,+,52138,97.19,97.20,0.474,"96.76, 97.86, 96.99"
7,+,+,-,52138,98.13,97.87,0.108,"97.77, 98.02, 97.82"
8,+,+,+,52138,98.03,97.80,0.118,"97.63, 97.88, 97.88"


<!--INTERP:factorial_table-->

Bảng giai thừa đang chờ số liệu thực thi.

In [7]:
def main_effect(rows, key):
    '''E_F = trung binh accuracy khi F bat, tru trung binh khi F tat (moi ben 4 to hop).'''
    on  = [r['accuracy'] for r in rows if r[key]]
    off = [r['accuracy'] for r in rows if not r[key]]
    return float(np.mean(on) - np.mean(off))


def interaction(rows, k1, k2):
    '''E_FG = nua hieu cua hai hieu ung don, do xem ich loi cua F co doi khi G bat hay khong.'''
    cell = lambda a, b: float(np.mean([r['accuracy'] for r in rows
                                       if r[k1] == a and r[k2] == b]))
    return 0.5 * ((cell(True, True) - cell(False, True))
                  - (cell(True, False) - cell(False, False)))


def effects_of(rows):
    return (
        {'padding': main_effect(rows, 'padding'),
         'he_init': main_effect(rows, 'he_init'),
         'lr_decay': main_effect(rows, 'lr_decay')},
        {'padding_he': interaction(rows, 'padding', 'he_init'),
         'padding_lr': interaction(rows, 'padding', 'lr_decay'),
         'he_lr':      interaction(rows, 'he_init', 'lr_decay')},
    )


MAIN_EFFECTS, INTERACTIONS = effects_of(FACT_ROWS)

# --- Thuoc do nhieu: do lech chuan gop cua 24 lan chay, roi suy ra sai so chuan cua hieu ung ---
SIGMA_RUN = float(np.sqrt(np.mean([r['accuracy_std'] ** 2 for r in FACT_ROWS])))
N_PER_SIDE = 4 * len(FACT_SEEDS)                      # 12 lan chay moi ben bat/tat
SE_EFFECT  = float(SIGMA_RUN * math.sqrt(2.0 / N_PER_SIDE))

VN = {'padding': 'Đệm viền (same padding)', 'he_init': 'Khởi tạo He Normal',
      'lr_decay': 'Lịch giảm tốc độ học',
      'padding_he': 'Đệm viền × He Normal', 'padding_lr': 'Đệm viền × Giảm LR',
      'he_lr': 'He Normal × Giảm LR'}

rank = sorted(MAIN_EFFECTS.items(), key=lambda kv: -kv[1])
print('HIỆU ỨNG CHÍNH, xếp hạng giảm dần (đơn vị: điểm phần trăm accuracy)')
print('-' * 78)
for i, (k, v) in enumerate(rank, 1):
    flag = 'vượt nhiễu' if abs(v) > 2 * SE_EFFECT else 'KHÔNG phân biệt được với nhiễu'
    print(f'  {i}. {VN[k]:<28} {v*100:+7.3f}   ({flag})')
print()
print('HIỆU ỨNG TƯƠNG TÁC BẬC HAI')
print('-' * 78)
for k, v in sorted(INTERACTIONS.items(), key=lambda kv: -abs(kv[1])):
    flag = 'vượt nhiễu' if abs(v) > 2 * SE_EFFECT else 'KHÔNG phân biệt được với nhiễu'
    print(f'     {VN[k]:<28} {v*100:+7.3f}   ({flag})')
print()
print('THƯỚC ĐO NHIỄU')
print('-' * 78)
print(f'  Độ lệch chuẩn gộp giữa các hạt giống trong cùng một tổ hợp : {SIGMA_RUN*100:.3f} điểm')
print(f'  Sai số chuẩn của một hiệu ứng (12 lần chạy mỗi bên)        : {SE_EFFECT*100:.3f} điểm')
print(f'  Ngưỡng 2 sai số chuẩn dùng để kết luận                     : {2*SE_EFFECT*100:.3f} điểm')

HIỆU ỨNG CHÍNH, xếp hạng giảm dần (đơn vị: điểm phần trăm accuracy)
------------------------------------------------------------------------------
  1. Khởi tạo He Normal            +0.489   (vượt nhiễu)
  2. Đệm viền (same padding)       +0.026   (KHÔNG phân biệt được với nhiễu)
  3. Lịch giảm tốc độ học          -0.204   (vượt nhiễu)

HIỆU ỨNG TƯƠNG TÁC BẬC HAI
------------------------------------------------------------------------------
     He Normal × Giảm LR           +0.078   (KHÔNG phân biệt được với nhiễu)
     Đệm viền × Giảm LR            +0.034   (KHÔNG phân biệt được với nhiễu)
     Đệm viền × He Normal          +0.007   (KHÔNG phân biệt được với nhiễu)

THƯỚC ĐO NHIỄU
------------------------------------------------------------------------------
  Độ lệch chuẩn gộp giữa các hạt giống trong cùng một tổ hợp : 0.207 điểm
  Sai số chuẩn của một hiệu ứng (12 lần chạy mỗi bên)        : 0.084 điểm
  Ngưỡng 2 sai số chuẩn dùng để kết luận                     : 0.169 điểm


<!--INTERP:main_effects-->

Hiệu ứng chính đang chờ số liệu thực thi.

In [8]:
print(f'Kiểm tra tính bền vững: chạy lại toàn bộ 8 tổ hợp ở lr0 = {LR_ROBUST} '
      f'(tốc độ học nền của bản Baseline)')
print('-' * 104)
t_rb = time.time()
FACT_ROWS_RB = run_factorial(LR_ROBUST)
print('-' * 104)
print(f'Thời gian: {time.time()-t_rb:.0f} giây')
print()

MAIN_RB, INTER_RB = effects_of(FACT_ROWS_RB)
SIGMA_RB = float(np.sqrt(np.mean([r['accuracy_std'] ** 2 for r in FACT_ROWS_RB])))
SE_RB = float(SIGMA_RB * math.sqrt(2.0 / N_PER_SIDE))

DF_ROBUST = pd.DataFrame([
    {'Yếu tố': VN[k],
     f'Hiệu ứng ở lr0={LR_PRIMARY} (điểm)': round(MAIN_EFFECTS[k] * 100, 3),
     f'Hiệu ứng ở lr0={LR_ROBUST} (điểm)': round(MAIN_RB[k] * 100, 3),
     'Cùng dấu': 'có' if MAIN_EFFECTS[k] * MAIN_RB[k] > 0 else 'KHÔNG'}
    for k in ('padding', 'he_init', 'lr_decay')] + [
    {'Yếu tố': VN[k],
     f'Hiệu ứng ở lr0={LR_PRIMARY} (điểm)': round(INTERACTIONS[k] * 100, 3),
     f'Hiệu ứng ở lr0={LR_ROBUST} (điểm)': round(INTER_RB[k] * 100, 3),
     'Cùng dấu': 'có' if INTERACTIONS[k] * INTER_RB[k] > 0 else 'KHÔNG'}
    for k in ('padding_he', 'padding_lr', 'he_lr')])

print('So sánh hiệu ứng giữa hai mức tốc độ học nền')
print(DF_ROBUST.to_string(index=False))
print()
print(f'Sai số chuẩn của hiệu ứng ở vòng kiểm tra: {SE_RB*100:.3f} điểm')
DF_ROBUST

Kiểm tra tính bền vững: chạy lại toàn bộ 8 tổ hợp ở lr0 = 0.001 (tốc độ học nền của bản Baseline)
--------------------------------------------------------------------------------------------------------


  [000] dem=0 He=0 giamLR=0 | test = 96.92 +/- 0.09 % | val = 96.93 % | epoch tot nhat ~ 15.0 | 106s


  [001] dem=0 He=0 giamLR=1 | test = 96.28 +/- 0.15 % | val = 96.33 % | epoch tot nhat ~ 14.7 | 92s


  [010] dem=0 He=1 giamLR=0 | test = 97.62 +/- 0.05 % | val = 97.71 % | epoch tot nhat ~ 14.0 | 94s


  [011] dem=0 He=1 giamLR=1 | test = 97.41 +/- 0.02 % | val = 97.33 % | epoch tot nhat ~ 14.0 | 85s


  [100] dem=1 He=0 giamLR=0 | test = 96.89 +/- 0.09 % | val = 96.74 % | epoch tot nhat ~ 14.7 | 78s


  [101] dem=1 He=0 giamLR=1 | test = 95.74 +/- 0.40 % | val = 95.53 % | epoch tot nhat ~ 15.3 | 110s


  [110] dem=1 He=1 giamLR=0 | test = 97.76 +/- 0.03 % | val = 97.94 % | epoch tot nhat ~ 14.3 | 107s


  [111] dem=1 He=1 giamLR=1 | test = 97.52 +/- 0.12 % | val = 97.71 % | epoch tot nhat ~ 14.3 | 91s
--------------------------------------------------------------------------------------------------------
Thời gian: 766 giây

So sánh hiệu ứng giữa hai mức tốc độ học nền
                 Yếu tố  Hiệu ứng ở lr0=0.002 (điểm)  Hiệu ứng ở lr0=0.001 (điểm) Cùng dấu
Đệm viền (same padding)                        0.026                       -0.081    KHÔNG
     Khởi tạo He Normal                        0.489                        1.123       có
   Lịch giảm tốc độ học                       -0.204                       -0.559       có
   Đệm viền × He Normal                        0.007                        0.204       có
     Đệm viền × Giảm LR                        0.034                       -0.138    KHÔNG
    He Normal × Giảm LR                        0.078                        0.336       có

Sai số chuẩn của hiệu ứng ở vòng kiểm tra: 0.068 điểm


,Yếu tố,Hiệu ứng ở lr0=0.002 (điểm),Hiệu ứng ở lr0=0.001 (điểm),Cùng dấu
0,Đệm viền (same padding),0.026,-0.081,KHÔNG
1,Khởi tạo He Normal,0.489,1.123,có
2,Lịch giảm tốc độ học,-0.204,-0.559,có
3,Đệm viền × He Normal,0.007,0.204,có
4,Đệm viền × Giảm LR,0.034,-0.138,KHÔNG
5,He Normal × Giảm LR,0.078,0.336,có


<!--INTERP:robustness-->

Kiểm tra bền vững đang chờ số liệu thực thi.

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6.2))

# ---- Panel trai: 8 to hop ----
ax = axes[0]
labels = [f"{'+' if r['padding'] else '-'} {'+' if r['he_init'] else '-'} "
          f"{'+' if r['lr_decay'] else '-'}" for r in FACT_ROWS]
vals = np.array([r['accuracy'] for r in FACT_ROWS]) * 100
errs = np.array([r['accuracy_std'] for r in FACT_ROWS]) * 100
colors = ['#2E7D32' if i == bundle else '#C62828' if i == plain else '#5B8DB8'
          for i in range(len(FACT_ROWS))]
bars = ax.bar(range(8), vals, yerr=errs, capsize=4, color=colors,
              edgecolor='black', linewidth=0.6)
rng_ = max(vals.max() - vals.min(), 0.5)
for b, v, e in zip(bars, vals, errs):
    ax.text(b.get_x() + b.get_width() / 2, v + e + rng_ * 0.04, f'{v:.2f}',
            ha='center', va='bottom', fontsize=9.5, fontweight='bold')
ax.set_xticks(range(8)); ax.set_xticklabels(labels, fontsize=11)
ax.set_xlabel('Tổ hợp theo thứ tự [Đệm viền] [He Normal] [Giảm LR],  "+" là bật, "-" là tắt')
ax.set_ylabel('Accuracy trên tập test (%)')
ax.set_title(f'Tám tổ hợp của thiết kế giai thừa $2^3$\n'
             f'{len(X_SUB_TR):,} ảnh huấn luyện, {FACT_EPOCHS} epoch, '
             f'trung bình {len(FACT_SEEDS)} hạt giống, thanh lỗi là độ lệch chuẩn',
             fontsize=11.5)
lo, hi = (vals - errs).min(), (vals + errs).max()
ax.set_ylim(lo - rng_ * 0.35, hi + rng_ * 0.22)
ax.axhline(vals[plain], color='#C62828', ls=':', lw=1.2, zorder=0)
handles = [plt.Rectangle((0, 0), 1, 1, fc='#C62828', ec='black', lw=0.6),
           plt.Rectangle((0, 0), 1, 1, fc='#2E7D32', ec='black', lw=0.6),
           plt.Rectangle((0, 0), 1, 1, fc='#5B8DB8', ec='black', lw=0.6)]
ax.legend(handles, ['Tắt cả ba (gói Baseline)', 'Bật cả ba (gói Improved)', 'Tổ hợp trung gian'],
          loc='lower right', fontsize=9.5)

# ---- Panel phai: hieu ung chinh va tuong tac ----
ax = axes[1]
keys = ['padding', 'he_init', 'lr_decay', 'padding_he', 'padding_lr', 'he_lr']
evals_ = np.array([MAIN_EFFECTS[k] if k in MAIN_EFFECTS else INTERACTIONS[k]
                   for k in keys]) * 100
short = ['Đệm viền', 'He Normal', 'Giảm LR', 'Đệm × He', 'Đệm × LR', 'He × LR']
cols = ['#1B5E20' if i < 3 else '#8E6C1F' for i in range(6)]
cols = [c if v >= 0 else '#B71C1C' for c, v in zip(cols, evals_)]
ax.barh(range(6), evals_, color=cols, edgecolor='black', linewidth=0.6)
span = max(abs(evals_).max(), 2 * SE_EFFECT * 100) * 1.45
ax.axvspan(-2 * SE_EFFECT * 100, 2 * SE_EFFECT * 100, color='grey', alpha=0.20, zorder=0,
           label=f'Dải nhiễu ±2 sai số chuẩn (±{2*SE_EFFECT*100:.3f} điểm)')
ax.axvline(0, color='black', lw=1.0)
for i, v in enumerate(evals_):
    off = span * 0.03
    ax.text(v + (off if v >= 0 else -off), i, f'{v:+.3f}',
            va='center', ha='left' if v >= 0 else 'right', fontsize=10, fontweight='bold')
ax.set_yticks(range(6)); ax.set_yticklabels(short, fontsize=11)
ax.invert_yaxis(); ax.set_xlim(-span, span)
ax.set_xlabel('Hiệu ứng (điểm phần trăm accuracy)')
ax.set_title('Hiệu ứng chính (ba thanh trên) và tương tác bậc hai (ba thanh dưới)\n'
             'Thanh nằm trong dải xám nghĩa là không phân biệt được với nhiễu hạt giống',
             fontsize=11.5)
ax.legend(loc='lower right', fontsize=9.5)

plt.tight_layout()
out = os.path.join(FIG_DIR, 'fig_ablation_factorial.png')
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.close(fig)
print('Đã lưu', out, '|', f'{os.path.getsize(out)/1024:.0f} KB')

Đã lưu ../reports/figures\fig_ablation_factorial.png | 124 KB


<!--INTERP:fig_factorial-->

Hình giai thừa đang chờ số liệu thực thi.

## 4. Phần B: đường cong theo cỡ dữ liệu và bóc tách khoảng cách

### 4.1 Vấn đề cần gỡ

Chương 6 và Chương 7 đưa ra hai con số không so sánh trực tiếp được với nhau. Mô hình NumPy thuần
huấn luyện trên tập con, còn mô hình framework huấn luyện trên toàn bộ nhánh train, và hai mô hình
lại có kiến trúc khác nhau, chạy trên hai tầng hiện thực khác nhau. Khi đặt hai accuracy cạnh nhau,
chênh lệch thu được là tổng của ba nguyên nhân chồng lên nhau mà không có cách nào gỡ ra từ chính
hai con số đó.

Đường cong theo cỡ dữ liệu gỡ được một trong ba nguyên nhân, bằng cách giữ **cố định** kiến trúc và
tầng hiện thực rồi chỉ thay đổi cỡ dữ liệu. Gọi $A(n)$ là accuracy của mô hình framework khi huấn
luyện trên $n$ ảnh. Với $n_{\text{nhỏ}}$ là cỡ tập con mà mô hình NumPy đã dùng và $n_{\text{đầy}}$
là cỡ lớn nhất khảo sát:

$$ \underbrace{A(n_{\text{đầy}}) - A_{\text{NumPy}}}_{\text{tổng khoảng cách}} \;=\; \underbrace{A(n_{\text{đầy}}) - A(n_{\text{nhỏ}})}_{\text{phần do dữ liệu}} \;+\; \underbrace{A(n_{\text{nhỏ}}) - A_{\text{NumPy}}}_{\text{phần còn lại}} $$

Phần thứ nhất là phép đo sạch, vì hai đại lượng trừ nhau chỉ khác nhau đúng một biến là cỡ dữ liệu.
Phần thứ hai thì **không** sạch: nó so sánh hai mô hình khác nhau cả về kiến trúc lẫn tầng hiện
thực, ở cùng một cỡ dữ liệu. Vì vậy báo cáo gọi nó là **phần còn lại** chứ không gọi là "phần do
kiến trúc", và coi nó là **cận trên** của ảnh hưởng kiến trúc. Thiết kế hiện tại hỗ trợ một phép
tách hai chiều, và báo cáo sẽ không tuyên bố một phép tách ba chiều mà nó không đo được.

### 4.2 Vì sao cố định ngân sách bước cập nhật chứ không cố định số epoch

Đây là chi tiết dễ làm hỏng toàn bộ kết luận. Nếu mọi điểm $n$ đều chạy cùng một số epoch, thì điểm
$n = 500$ chỉ nhận khoảng 60 bước cập nhật trong khi điểm $n = 40\,000$ nhận gần 5 000 bước. Đường
cong thu được sẽ dốc, nhưng độ dốc đó **trộn lẫn** "ít dữ liệu" với "ít được huấn luyện", và phần
"do dữ liệu" trong phép bóc tách sẽ bị thổi phồng.

Notebook vì thế cấp cho **mọi** điểm $n$ cùng một ngân sách bước cập nhật, đo validation đều đặn
trong suốt quá trình và giữ lại checkpoint tốt nhất. Ở $n$ nhỏ, ngân sách này tương đương hàng trăm
lượt duyệt dữ liệu và mô hình chắc chắn sẽ quá khớp, nhưng việc chọn checkpoint theo validation
đúng là cơ chế xử lý tình huống đó: điểm $n$ nhỏ sẽ dừng lại ở mức mà dữ liệu của nó cho phép, chứ
không phải ở mức mà thời lượng huấn luyện cho phép.

### 4.3 Giao thức

| Hạng mục | Giá trị |
|---|---|
| Kiến trúc | `MnistCNN` nạp từ `mnist/models/mnist_cnn_def.py`, đúng mô hình của Chương 7 |
| Dãy cỡ dữ liệu | 500, 1 000, 2 000, 5 000, 10 000, cỡ tập con NumPy, 20 000, 40 000 |
| Ngân sách | 2 000 bước cập nhật, batch 128, Adam $10^{-3}$ |
| Chọn mô hình | Đo validation mỗi 100 bước, giữ checkpoint có validation cao nhất |
| Validation | 12 000 ảnh cố định, dùng chung cho mọi điểm |
| Hạt giống | 42, 43, 44; mỗi hạt giống **rút lại** tập con nên dải bóng bao gồm cả nhiễu lấy mẫu |

Cỡ tập con NumPy được **chèn thêm** vào dãy khảo sát, vì nếu không có nó thì $A(n_{\text{nhỏ}})$ chỉ
có thể nội suy, mà một đại lượng đi thẳng vào kết luận thì nên được đo chứ không nên nội suy. Ở hạt
giống 42, tập con tại điểm này trùng khít từng ảnh với tập con mà Chương 6 đã dùng.

In [10]:
LC_STEPS, LC_BATCH, LC_LR, LC_EVAL_EVERY = 2000, 128, 1e-3, 100
LC_SEEDS = [42, 43, 44]
N_VALUES = sorted(set([500, 1000, 2000, 5000, 10000, 20000, 40000] + [N_SMALL]))


def subset_indices(n, seed):
    '''Rut n chi so co phan tang tu nhanh train 48 000 anh. Voi seed=42 va n=N_SMALL,
    lenh nay tai lap chinh xac tap con ma Chuong 6 da dung.'''
    if n >= len(y_tr_full):
        return np.arange(len(y_tr_full))
    idx, _ = train_test_split(np.arange(len(y_tr_full)), train_size=n,
                              stratify=y_tr_full, random_state=seed)
    return idx


# Bang chung: tai n = N_SMALL, hat giong 42 cho dung tap con cua Chuong 6
_chk = subset_indices(N_SMALL, 42)
assert np.array_equal(np.sort(_chk), np.sort(sub_tr)), 'Tap con khong trung voi Chuong 6'
print(f'Kiem chung: tap con {N_SMALL} anh o hat giong 42 trung khop tung anh voi Chuong 6.')
print(f'Day co du lieu khao sat: {N_VALUES}')
print(f'Ngan sach: {LC_STEPS} buoc x batch {LC_BATCH} cho MOI diem, '
      f'do validation moi {LC_EVAL_EVERY} buoc')
print(f'So lan huan luyen: {len(N_VALUES)} co x {len(LC_SEEDS)} hat giong = '
      f'{len(N_VALUES)*len(LC_SEEDS)}')
print('-' * 104)

LC_RAW = {n: [] for n in N_VALUES}
t_lc = time.time()
for sd in LC_SEEDS:
    for n in N_VALUES:
        idx = torch.from_numpy(subset_indices(n, sd)).to(DEVICE)
        Xn, yn = X_TR_FULL[idx], Y_TR_FULL[idx]
        torch.manual_seed(sd)
        model = MnistCNN()
        best, _, el = train_steps(model, Xn, yn, X_VA_FULL, Y_VA_FULL,
                                  total_steps=LC_STEPS, bs=LC_BATCH, lr0=LC_LR,
                                  eval_every=LC_EVAL_EVERY, seed=sd)
        model.load_state_dict(best['state'])
        _, te = evaluate(model, X_TE, Y_TE)
        LC_RAW[n].append(float(te))
        print(f'  hat giong {sd} | n = {n:>6,} | val tot nhat {best["acc"]*100:5.2f} % '
              f'tai buoc {best["step"]:>4} | test {te*100:5.2f} % | {el:5.1f}s')
    print('  ' + '-' * 100)
print(f'Tong thoi gian duong cong: {time.time()-t_lc:.0f} giay')

Kiem chung: tap con 12000 anh o hat giong 42 trung khop tung anh voi Chuong 6.
Day co du lieu khao sat: [500, 1000, 2000, 5000, 10000, 12000, 20000, 40000]
Ngan sach: 2000 buoc x batch 128 cho MOI diem, do validation moi 100 buoc
So lan huan luyen: 8 co x 3 hat giong = 24
--------------------------------------------------------------------------------------------------------


  hat giong 42 | n =    500 | val tot nhat 95.85 % tai buoc 1900 | test 95.82 % |  63.1s


  hat giong 42 | n =  1,000 | val tot nhat 96.70 % tai buoc 2000 | test 96.89 % |  68.3s


  hat giong 42 | n =  2,000 | val tot nhat 97.78 % tai buoc 1800 | test 97.87 % |  61.2s


  hat giong 42 | n =  5,000 | val tot nhat 98.30 % tai buoc 1400 | test 98.56 % |  54.8s


...

  hat giong 44 | n =  5,000 | val tot nhat 98.29 % tai buoc 1400 | test 98.32 % |  31.7s


  hat giong 44 | n = 10,000 | val tot nhat 98.57 % tai buoc 1800 | test 98.68 % |  35.0s


  hat giong 44 | n = 12,000 | val tot nhat 98.71 % tai buoc 1900 | test 98.75 % |  35.2s


  hat giong 44 | n = 20,000 | val tot nhat 98.86 % tai buoc 2000 | test 98.91 % |  34.8s


  hat giong 44 | n = 40,000 | val tot nhat 98.88 % tai buoc 1900 | test 98.92 % |  30.9s
  ----------------------------------------------------------------------------------------------------
Tong thoi gian duong cong: 1006 giay


In [11]:
LC_MEAN = [float(np.mean(LC_RAW[n])) for n in N_VALUES]
LC_STD  = [float(np.std(LC_RAW[n], ddof=0)) for n in N_VALUES]

DF_LC = pd.DataFrame({
    'n huấn luyện': N_VALUES,
    'Accuracy trung bình (%)': [round(m * 100, 2) for m in LC_MEAN],
    'Độ lệch chuẩn (%)': [round(s * 100, 3) for s in LC_STD],
    'Ba hạt giống (%)': [', '.join(f'{a*100:.2f}' for a in LC_RAW[n]) for n in N_VALUES],
    'Lỗi còn lại (%)': [round((1 - m) * 100, 2) for m in LC_MEAN],
})
DF_LC['Ghi chú'] = ['cỡ tập con NumPy' if n == N_SMALL else
                    ('cỡ lớn nhất khảo sát' if n == max(N_VALUES) else '') for n in N_VALUES]

print('Accuracy theo cỡ dữ liệu huấn luyện (kiến trúc framework, ngân sách bước cố định)')
print()
print(DF_LC.to_string(index=False))
print()
# Kiem tra quy luat luy thua: loi con lai giam tuyen tinh theo log n hay khong
_lg = np.log10(N_VALUES); _le = np.log10([1 - m for m in LC_MEAN])
_slope, _icpt = np.polyfit(_lg, _le, 1)
_r = float(np.corrcoef(_lg, _le)[0, 1])
print(f'Hồi quy log(lỗi) theo log(n): độ dốc = {_slope:.3f}, hệ số tương quan = {_r:.4f}')
print(f'Diễn giải: mỗi lần nhân đôi lượng dữ liệu, lỗi còn lại nhân với '
      f'{2**_slope:.3f} (giảm {(1-2**_slope)*100:.1f} %)')
print()
print(f'Từ n = {N_VALUES[0]:,} lên n = {max(N_VALUES):,} (gấp {max(N_VALUES)//N_VALUES[0]} lần), '
      f'accuracy tăng {(LC_MEAN[-1]-LC_MEAN[0])*100:.2f} điểm phần trăm')
print(f'Từ n = {N_SMALL:,} lên n = {max(N_VALUES):,} (gấp {max(N_VALUES)/N_SMALL:.1f} lần), '
      f'accuracy chỉ tăng '
      f'{(LC_MEAN[-1]-LC_MEAN[N_VALUES.index(N_SMALL)])*100:.2f} điểm phần trăm')
DF_LC

Accuracy theo cỡ dữ liệu huấn luyện (kiến trúc framework, ngân sách bước cố định)

 n huấn luyện  Accuracy trung bình (%)  Độ lệch chuẩn (%)    Ba hạt giống (%)  Lỗi còn lại (%)              Ghi chú
          500                    95.56              0.240 95.82, 95.24, 95.61             4.44                     
         1000                    96.89              0.078 96.89, 96.80, 96.99             3.11                     
         2000                    97.70              0.135 97.87, 97.68, 97.54             2.30                     
         5000                    98.35              0.157 98.56, 98.18, 98.32             1.65                     
        10000                    98.68              0.053 98.74, 98.61, 98.68             1.32                     
        12000                    98.77              0.115 98.92, 98.64, 98.75             1.23     cỡ tập con NumPy
        20000                    98.74              0.148 98.76, 98.55, 98.91             1.26           

,n huấn luyện,Accuracy trung bình (%),Độ lệch chuẩn (%),Ba hạt giống (%),Lỗi còn lại (%),Ghi chú
0,500,95.56,0.240,"95.82, 95.24, 95.61",4.44,
1,1000,96.89,0.078,"96.89, 96.80, 96.99",3.11,
2,2000,97.70,0.135,"97.87, 97.68, 97.54",2.30,
3,5000,98.35,0.157,"98.56, 98.18, 98.32",1.65,
4,10000,98.68,0.053,"98.74, 98.61, 98.68",1.32,
5,12000,98.77,0.115,"98.92, 98.64, 98.75",1.23,cỡ tập con NumPy
6,20000,98.74,0.148,"98.76, 98.55, 98.91",1.26,
7,40000,98.86,0.060,"98.89, 98.78, 98.92",1.14,cỡ lớn nhất khảo sát


<!--INTERP:lc_table-->

Bảng đường cong đang chờ số liệu thực thi.

### 4.4 Đọc lại số liệu tham chiếu ngay trước khi dùng

Tệp `mnist/reports/metrics_mnist.json` đang được huấn luyện lại trên GPU song song với notebook
này, nên bản đã đọc ở Mục 2 có thể đã cũ. Ba con số đi thẳng vào phép bóc tách là accuracy của CNN
NumPy bản Improved, của PyTorch và của Keras, vì vậy chúng được **đọc lại một lần duy nhất tại đây**
rồi dùng chung cho cả hình vẽ lẫn phép tính. Đọc một lần chung như vậy còn bảo đảm hình và bảng
không lỡ tham chiếu hai phiên bản khác nhau của cùng một tệp.

In [12]:
MNIST_REF, MNIST_REF_MTIME = load_mnist_ref()

A_NUMPY      = float(MNIST_REF['models']['numpy_improved']['accuracy'])
A_NUMPY_BASE = float(MNIST_REF['models']['numpy_baseline']['accuracy'])
A_PT_CHAPTER = float(MNIST_REF['models']['pytorch']['accuracy'])
A_TF_CHAPTER = float(MNIST_REF['models']['tensorflow']['accuracy'])
N_TRAIN_FULL = int(_dig(MNIST_REF, 'n_train', len(y_tr_full)))
PT_FRAMEWORK = MNIST_REF['models']['pytorch'].get('framework', 'PyTorch')
TF_FRAMEWORK = MNIST_REF['models']['tensorflow'].get('framework', 'TensorFlow')

print(f'Phiên bản metrics_mnist.json dùng cho phép bóc tách: sửa lúc {MNIST_REF_MTIME}')
print('-' * 84)
print(f'  numpy_baseline : {A_NUMPY_BASE*100:6.2f} %   '
      f'({MNIST_REF["models"]["numpy_baseline"].get("framework", "")})')
print(f'  numpy_improved : {A_NUMPY*100:6.2f} %   '
      f'({MNIST_REF["models"]["numpy_improved"].get("framework", "")})')
print(f'  pytorch        : {A_PT_CHAPTER*100:6.2f} %   ({PT_FRAMEWORK})')
print(f'  tensorflow     : {A_TF_CHAPTER*100:6.2f} %   ({TF_FRAMEWORK})')
print(f'  n_train của Chương 7 : {N_TRAIN_FULL:,} ảnh')

Doc ../../mnist/reports/metrics_mnist.json (sua luc 2026-09-16 14:43:55) -> du ca 4 mo hinh


Phiên bản metrics_mnist.json dùng cho phép bóc tách: sửa lúc 2026-09-16 14:43:55
------------------------------------------------------------------------------------
  numpy_baseline :  97.03 %   (NumPy From Scratch (Baseline))
  numpy_improved :  98.23 %   (NumPy From Scratch (Improved))
  pytorch        :  99.06 %   (PyTorch 2.13.0+cu126 (GPU))
  tensorflow     :  99.02 %   (TensorFlow 2.21.0 / Keras 3.15.1 (CPU))
  n_train của Chương 7 : 48,000 ảnh


In [13]:
i_small, i_full = N_VALUES.index(N_SMALL), len(N_VALUES) - 1

fig, ax = plt.subplots(figsize=(11.5, 6.8))
mean_pct = np.array(LC_MEAN) * 100
std_pct = np.array(LC_STD) * 100

ax.fill_between(N_VALUES, mean_pct - std_pct, mean_pct + std_pct,
                color='#1F77B4', alpha=0.22,
                label='Dải ±1 độ lệch chuẩn trên 3 hạt giống')
ax.plot(N_VALUES, mean_pct, 'o-', color='#1F77B4', lw=2.2, ms=7,
        label='CNN framework, accuracy trung bình', zorder=3)
for n, m in zip(N_VALUES, mean_pct):
    ax.annotate(f'{m:.2f}', (n, m), textcoords='offset points', xytext=(0, 11),
                ha='center', fontsize=9)

ax.axvline(N_SMALL, color='#C62828', ls='--', lw=1.6, zorder=2)
ax.axhline(A_NUMPY * 100, color='#2E7D32', ls=':', lw=1.8,
           label=f'CNN NumPy Improved, Chương 6 ({A_NUMPY*100:.2f} %)')
ax.axhline(A_NUMPY_BASE * 100, color='#8E6C1F', ls=':', lw=1.4,
           label=f'CNN NumPy Baseline, Chương 6 ({A_NUMPY_BASE*100:.2f} %)')

ax.scatter([N_SMALL], [mean_pct[i_small]], s=190, facecolor='none',
           edgecolor='#C62828', lw=2.4, zorder=4)
ax.scatter([N_VALUES[-1]], [mean_pct[i_full]], s=190, facecolor='none',
           edgecolor='#4A148C', lw=2.4, zorder=4)

ax.set_xscale('log')
ax.set_xticks(N_VALUES)
ax.set_xticklabels([f'{n:,}' for n in N_VALUES], fontsize=10)
ax.get_xaxis().set_minor_formatter(plt.NullFormatter())
ax.set_xlabel('Số ảnh huấn luyện n (thang logarit)')
ax.set_ylabel('Accuracy trên tập test (%)')
ax.set_title('Đường cong accuracy theo cỡ dữ liệu huấn luyện\n'
             f'Kiến trúc CNN framework giữ nguyên, ngân sách {LC_STEPS:,} bước cập nhật '
             f'cho mọi điểm, 3 hạt giống', fontsize=12.5)
y_lo = min((mean_pct - std_pct).min(), A_NUMPY_BASE * 100) - 0.55
y_hi = max(mean_pct.max(), A_NUMPY * 100) + 0.75
ax.set_ylim(y_lo, y_hi)
ax.annotate(f'Cỡ tập con mô hình NumPy dùng\nn = {N_SMALL:,}',
            xy=(N_SMALL, y_lo + (y_hi - y_lo) * 0.10),
            xytext=(N_SMALL * 0.28, y_lo + (y_hi - y_lo) * 0.04),
            fontsize=10, color='#C62828',
            arrowprops=dict(arrowstyle='->', color='#C62828', lw=1.3))
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, which='major', alpha=0.35)

plt.tight_layout()
out = os.path.join(FIG_DIR, 'fig_learning_curve.png')
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.close(fig)
print('Đã lưu', out, '|', f'{os.path.getsize(out)/1024:.0f} KB')
print(f'Điểm đánh dấu đỏ  : n = {N_SMALL:,}  -> {mean_pct[i_small]:.2f} %  (A_nhỏ)')
print(f'Điểm đánh dấu tím : n = {N_VALUES[-1]:,} -> {mean_pct[i_full]:.2f} %  (A_đầy)')

Đã lưu ../reports/figures\fig_learning_curve.png | 138 KB
Điểm đánh dấu đỏ  : n = 12,000  -> 98.77 %  (A_nhỏ)
Điểm đánh dấu tím : n = 40,000 -> 98.86 %  (A_đầy)


<!--INTERP:fig_lc-->

Hình đường cong đang chờ số liệu thực thi.

### 4.5 Bóc tách khoảng cách

Ba đại lượng đi vào phép bóc tách, tất cả đều đo trên **cùng** 10 000 ảnh test:

- $A_{\text{NumPy}}$: accuracy của CNN NumPy bản Improved, đọc từ `mnist/reports/metrics_mnist.json`.
- $A_{\text{nhỏ}}$: accuracy của CNN framework tại đúng cỡ dữ liệu mà mô hình NumPy đã dùng.
- $A_{\text{đầy}}$: accuracy của CNN framework tại cỡ dữ liệu lớn nhất mà notebook khảo sát.

Từ đó:

$$ \text{phần do dữ liệu} = A_{\text{đầy}} - A_{\text{nhỏ}}, \qquad \text{phần còn lại} = A_{\text{nhỏ}} - A_{\text{NumPy}} $$

Một lưu ý về $A_{\text{đầy}}$: cỡ lớn nhất mà notebook khảo sát là 40 000 ảnh, trong khi Chương 7
huấn luyện trên toàn bộ 48 000 ảnh của nhánh train. Nói cách khác $A_{\text{đầy}}$ ở đây hơi thấp
hơn giá trị mà dữ liệu đầy đủ có thể đạt, nên **phần do dữ liệu bị ước lượng thiếu một chút** chứ
không bị thổi phồng. Sai lệch theo chiều thận trọng như vậy chấp nhận được, và phần diễn giải sẽ
đối chiếu $A_{\text{đầy}}$ với con số mà Chương 7 đã báo cáo.

In [14]:
A_SMALL = LC_MEAN[i_small]
A_FULL  = LC_MEAN[i_full]
A_SMALL_STD, A_FULL_STD = LC_STD[i_small], LC_STD[i_full]
N_FULL = N_VALUES[-1]

TOTAL_GAP  = A_FULL - A_NUMPY
DATA_COMP  = A_FULL - A_SMALL
RESID_COMP = A_SMALL - A_NUMPY
data_share  = DATA_COMP / TOTAL_GAP if TOTAL_GAP != 0 else float('nan')
resid_share = RESID_COMP / TOTAL_GAP if TOTAL_GAP != 0 else float('nan')

print('BA ĐẠI LƯỢNG ĐẦU VÀO')
print('-' * 84)
print(f'  A_NumPy (CNN NumPy Improved, Chương 6, n = {N_SMALL:,})   : {A_NUMPY*100:6.2f} %')
print(f'  A_nhỏ   (CNN framework, n = {N_SMALL:,})                  : {A_SMALL*100:6.2f} % '
      f'+/- {A_SMALL_STD*100:.2f}')
print(f'  A_đầy   (CNN framework, n = {N_FULL:,})                  : {A_FULL*100:6.2f} % '
      f'+/- {A_FULL_STD*100:.2f}')
print()
print('BÓC TÁCH KHOẢNG CÁCH')
print('-' * 84)
print(f'  Tổng khoảng cách  A_đầy - A_NumPy   : {TOTAL_GAP*100:+6.2f} điểm phần trăm')
print(f'  Phần do dữ liệu   A_đầy - A_nhỏ     : {DATA_COMP*100:+6.2f} điểm '
      f'({data_share*100:5.1f} % tổng)')
print(f'  Phần còn lại      A_nhỏ - A_NumPy   : {RESID_COMP*100:+6.2f} điểm '
      f'({resid_share*100:5.1f} % tổng)')
print()
print('ĐỐI CHIẾU VỚI SỐ LIỆU ĐÃ BÁO CÁO')
print('-' * 84)
print(f'  PyTorch Chương 7 (n = {N_TRAIN_FULL:,})      : {A_PT_CHAPTER*100:6.2f} %')
print(f'  Keras   Chương 7 (n = {N_TRAIN_FULL:,})      : {A_TF_CHAPTER*100:6.2f} %')
print(f'  A_đầy của notebook này (n = {N_FULL:,}) : {A_FULL*100:6.2f} %')
print(f'  Chênh lệch so với PyTorch Chương 7      : {(A_FULL-A_PT_CHAPTER)*100:+.2f} điểm')
print()
print('KIỂM TRA ĐỘ TIN CẬY CỦA PHÉP BÓC TÁCH')
print('-' * 84)
_se_data = math.sqrt(A_FULL_STD**2 + A_SMALL_STD**2) / math.sqrt(len(LC_SEEDS))
print(f'  Sai số chuẩn xấp xỉ của phần do dữ liệu : {_se_data*100:.3f} điểm')
print(f'  Phần do dữ liệu {"vượt" if abs(DATA_COMP) > 2*_se_data else "KHÔNG vượt"} '
      f'ngưỡng 2 sai số chuẩn ({2*_se_data*100:.3f} điểm)')

BA ĐẠI LƯỢNG ĐẦU VÀO
------------------------------------------------------------------------------------
  A_NumPy (CNN NumPy Improved, Chương 6, n = 12,000)   :  98.23 %
  A_nhỏ   (CNN framework, n = 12,000)                  :  98.77 % +/- 0.12
  A_đầy   (CNN framework, n = 40,000)                  :  98.86 % +/- 0.06

BÓC TÁCH KHOẢNG CÁCH
------------------------------------------------------------------------------------
  Tổng khoảng cách  A_đầy - A_NumPy   :  +0.63 điểm phần trăm
  Phần do dữ liệu   A_đầy - A_nhỏ     :  +0.09 điểm ( 14.7 % tổng)
  Phần còn lại      A_nhỏ - A_NumPy   :  +0.54 điểm ( 85.3 % tổng)

ĐỐI CHIẾU VỚI SỐ LIỆU ĐÃ BÁO CÁO
------------------------------------------------------------------------------------
  PyTorch Chương 7 (n = 48,000)      :  99.06 %
  Keras   Chương 7 (n = 48,000)      :  99.02 %
  A_đầy của notebook này (n = 40,000) :  98.86 %
  Chênh lệch so với PyTorch Chương 7      : -0.20 điểm

KIỂM TRA ĐỘ TIN CẬY CỦA PHÉP BÓC TÁCH
-----------------

<!--INTERP:gap-->

Bóc tách khoảng cách đang chờ số liệu thực thi.

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(14.5, 6.3),
                         gridspec_kw={'width_ratios': [1, 1.55]})

# ---- Panel trai: thanh xep chong tach tong khoang cach ----
ax = axes[0]
d_pp, r_pp, t_pp = DATA_COMP * 100, RESID_COMP * 100, TOTAL_GAP * 100
if d_pp >= 0 and r_pp >= 0:
    ax.bar([0], [d_pp], width=0.52, color='#1F77B4', edgecolor='black', lw=0.7,
           label=f'Phần do cỡ dữ liệu  ({d_pp:.2f} điểm, {data_share*100:.0f} %)')
    ax.bar([0], [r_pp], width=0.52, bottom=[d_pp], color='#D98C1F', edgecolor='black', lw=0.7,
           label=f'Phần còn lại: kiến trúc và tầng hiện thực  ({r_pp:.2f} điểm, '
                 f'{resid_share*100:.0f} %)')
    ax.text(0, d_pp / 2, f'{d_pp:.2f}', ha='center', va='center',
            color='white', fontweight='bold', fontsize=12)
    ax.text(0, d_pp + r_pp / 2, f'{r_pp:.2f}', ha='center', va='center',
            color='white', fontweight='bold', fontsize=12)
    ax.text(0, t_pp * 1.045, f'Tổng {t_pp:.2f} điểm', ha='center', va='bottom',
            fontsize=11.5, fontweight='bold')
    ax.set_ylim(0, t_pp * 1.28)
    ax.set_xticks([])
    ax.legend(loc='upper center', fontsize=9, framealpha=0.95)
else:
    # Mot thanh phan mang dau am nen khong xep chong duoc, chuyen sang hai thanh rieng
    ax.bar([-0.3, 0.3], [d_pp, r_pp], width=0.42,
           color=['#1F77B4', '#D98C1F'], edgecolor='black', lw=0.7)
    ax.axhline(0, color='black', lw=1)
    ax.set_xticks([-0.3, 0.3])
    ax.set_xticklabels(['Phần do cỡ dữ liệu', 'Phần còn lại'], fontsize=10)
    for x, v in ((-0.3, d_pp), (0.3, r_pp)):
        ax.text(x, v, f'{v:+.2f}', ha='center',
                va='bottom' if v >= 0 else 'top', fontsize=11, fontweight='bold')
    ax.text(0.5, 0.02, 'Một thành phần mang dấu âm nên không xếp chồng được',
            transform=ax.transAxes, ha='center', fontsize=9.5, style='italic')
ax.set_ylabel('Điểm phần trăm accuracy')
ax.set_title('Tách tổng khoảng cách NumPy với framework\n'
             'thành phần do dữ liệu và phần còn lại', fontsize=12)

# ---- Panel phai: ba muc accuracy ----
ax = axes[1]
names = [f'CNN NumPy Improved\nn = {N_SMALL:,}',
         f'CNN framework\nn = {N_SMALL:,}',
         f'CNN framework\nn = {N_FULL:,}']
vals3 = np.array([A_NUMPY, A_SMALL, A_FULL]) * 100
errs3 = np.array([0.0, A_SMALL_STD, A_FULL_STD]) * 100
bars = ax.bar(range(3), vals3, yerr=errs3, capsize=5, width=0.55,
              color=['#2E7D32', '#D98C1F', '#1F77B4'], edgecolor='black', lw=0.7)
for b, v in zip(bars, vals3):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.035, f'{v:.2f} %',
            ha='center', va='bottom', fontsize=11.5, fontweight='bold')

rng3 = max(vals3.max() - vals3.min(), 0.4)
ax.set_ylim(vals3.min() - rng3 * 0.9, vals3.max() + rng3 * 0.75)
y_a, y_b = vals3[0], vals3[1]
ax.annotate('', xy=(1, y_b), xytext=(0, y_a),
            arrowprops=dict(arrowstyle='->', color='#D98C1F', lw=2.0,
                            connectionstyle='arc3,rad=-0.25'))
ax.text(0.5, max(y_a, y_b) + rng3 * 0.30, f'phần còn lại\n{r_pp:+.2f} điểm',
        ha='center', fontsize=10, color='#8a5a12', fontweight='bold')
ax.annotate('', xy=(2, vals3[2]), xytext=(1, y_b),
            arrowprops=dict(arrowstyle='->', color='#1F77B4', lw=2.0,
                            connectionstyle='arc3,rad=-0.25'))
ax.text(1.5, max(y_b, vals3[2]) + rng3 * 0.30, f'phần do dữ liệu\n{d_pp:+.2f} điểm',
        ha='center', fontsize=10, color='#14507d', fontweight='bold')

ax.set_xticks(range(3)); ax.set_xticklabels(names, fontsize=10)
ax.set_ylabel('Accuracy trên tập test (%)')
ax.set_title('Ba mức accuracy tạo nên phép bóc tách\n'
             'Bước thứ nhất đổi cả kiến trúc lẫn tầng hiện thực, bước thứ hai chỉ đổi cỡ dữ liệu',
             fontsize=12)

plt.tight_layout()
out = os.path.join(FIG_DIR, 'fig_gap_decomposition.png')
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='white')
plt.close(fig)
print('Đã lưu', out, '|', f'{os.path.getsize(out)/1024:.0f} KB')

Đã lưu ../reports/figures\fig_gap_decomposition.png | 122 KB


<!--INTERP:fig_gap-->

Hình bóc tách đang chờ số liệu thực thi.

## 5. Ghi tệp metrics

Toàn bộ số liệu được ghi ra `analysis/reports/metrics_ablation.json` theo đúng lược đồ của hợp
đồng. Các khóa bắt buộc giữ nguyên tên và giữ thang $[0,1]$. Một số khóa bổ sung được thêm vào,
đều mang hậu tố rõ nghĩa: `accuracy_std` và `accuracy_seeds` cho từng tổ hợp, `*_pp` cho các đại
lượng quy đổi sang điểm phần trăm, `data_share` và `residual_share` cho tỉ lệ đóng góp, cùng khối
`robustness` chứa vòng chạy kiểm tra ở tốc độ học nền thứ hai.

In [16]:
metrics = {
    'factorial': {
        'combinations': FACT_ROWS,
        'main_effects': MAIN_EFFECTS,
        'interactions': INTERACTIONS,
        'n_train': int(len(X_SUB_TR)),
        'epochs': int(FACT_EPOCHS),
        'n_val': int(len(X_SUB_VA)),
        'n_test': int(len(X_TE)),
        'batch_size': int(FACT_BATCH),
        'lr0': float(LR_PRIMARY),
        'lr_decay_gamma': float(DECAY_GAMMA),
        'lr_decay_every': int(DECAY_EVERY),
        'seeds': FACT_SEEDS,
        'seed_sigma_pooled': SIGMA_RUN,
        'effect_standard_error': SE_EFFECT,
        'main_effects_pp': {k: v * 100 for k, v in MAIN_EFFECTS.items()},
        'interactions_pp': {k: v * 100 for k, v in INTERACTIONS.items()},
        'ranking': [k for k, _ in rank],
        'robustness': {
            'lr0': float(LR_ROBUST),
            'combinations': FACT_ROWS_RB,
            'main_effects': MAIN_RB,
            'interactions': INTER_RB,
            'effect_standard_error': SE_RB,
        },
    },
    'learning_curve': {
        'n_values': [int(n) for n in N_VALUES],
        'mean': LC_MEAN,
        'std': LC_STD,
        'seeds': LC_SEEDS,
        'per_seed': {str(n): LC_RAW[n] for n in N_VALUES},
        'total_steps': int(LC_STEPS),
        'batch_size': int(LC_BATCH),
        'lr': float(LC_LR),
        'eval_every': int(LC_EVAL_EVERY),
        'architecture': 'MnistCNN (mnist/models/mnist_cnn_def.py)',
        'log_error_slope': float(_slope),
    },
    'gap_decomposition': {
        'a_numpy': A_NUMPY,
        'a_small': A_SMALL,
        'a_full': A_FULL,
        'n_small': int(N_SMALL),
        'n_full': int(N_FULL),
        'data_component': DATA_COMP,
        'residual_component': RESID_COMP,
        'total_gap': TOTAL_GAP,
        'data_component_pp': DATA_COMP * 100,
        'residual_component_pp': RESID_COMP * 100,
        'total_gap_pp': TOTAL_GAP * 100,
        'data_share': data_share,
        'residual_share': resid_share,
        'a_small_std': A_SMALL_STD,
        'a_full_std': A_FULL_STD,
        'a_numpy_baseline_chapter6': A_NUMPY_BASE,
        'a_pytorch_chapter7': A_PT_CHAPTER,
        'a_tensorflow_chapter7': A_TF_CHAPTER,
        'n_train_chapter7': int(N_TRAIN_FULL),
        'source_metrics_mtime': MNIST_REF_MTIME,
    },
    'notes': (
        'Notebook 06 (mien analysis). Phan A: thiet ke giai thua day du 2^3 tren ban sao PyTorch '
        f'cua CNN NumPy Chuong 6 ({len(X_SUB_TR)} anh, {FACT_EPOCHS} epoch, batch {FACT_BATCH}, '
        f'Adam lr0={LR_PRIMARY}), moi to hop lap lai voi 3 hat giong; khoa accuracy la trung binh '
        '3 hat giong, accuracy_seeds giu day du gia tri tung lan chay. So tham so cua ban sao '
        'trung khop tuyet doi voi numpy_baseline (27562) va numpy_improved (52138) cua Chuong 6. '
        'SAI LECH CO CHU DINH so voi Chuong 6: batch 128 thay vi 64 (o batch 64 mo hinh qua nho '
        'nen thoi gian bi chi phi goi nhan GPU chi phoi, do duoc 4,13 s/epoch so voi 1,13 s/epoch); '
        'batch duoc giu nguyen cho ca 8 to hop nen khong lam lech phep so sanh giua cac to hop, '
        'chi lam accuracy tuyet doi thap hon Chuong 6 mot chut. PHAT HIEN VE THIET KE: goi '
        '"Improved" cua Chuong 6 thuc ra doi BON thu chu khong phai ba, vi no con nang toc do hoc '
        'nen tu 1e-3 len 2e-3; notebook giu co dinh toc do hoc nen trong ca 8 to hop va chay lai '
        'toan bo thiet ke o ca hai muc (khoi robustness) de kiem tra tinh ben vung cua ket luan. '
        'Phan B: duong cong theo co du lieu dung DUNG kien truc MnistCNN cua Chuong 7, cap ngan '
        f'sach {LC_STEPS} buoc cap nhat cho MOI diem n thay vi co dinh so epoch, de do doc cua '
        'duong cong khong tron lan "it du lieu" voi "it duoc huan luyen"; checkpoint chon theo '
        f'validation do moi {LC_EVAL_EVERY} buoc. Co tap con NumPy ({N_SMALL}) duoc chen them vao '
        'day khao sat de A_nho duoc DO chu khong phai noi suy; o hat giong 42 tap con nay trung '
        'khop tung anh voi tap con Chuong 6. Moi hat giong rut lai tap con nen dai bong bao gom ca '
        'nhieu lay mau lan nhieu khoi tao. GIOI HAN CUA PHEP BOC TACH: thiet ke chi ho tro tach '
        'HAI phan chu khong phai ba; phan con lai (A_nho - A_numpy) van gop chung kien truc voi '
        'tang hien thuc nen no la CAN TREN cua anh huong kien truc, khong phai phep do rieng kien '
        f'truc. A_day do tai n={N_FULL} trong khi Chuong 7 huan luyen tren {N_TRAIN_FULL} anh, nen '
        'phan do du lieu bi uoc luong THIEU mot chut chu khong bi thoi phong. Vi Chuong 6 va 7 dang '
        'duoc chay lai song song, metrics_mnist.json duoc doc lai ngay truoc cho dung (phien ban '
        f'{MNIST_REF_MTIME}, ghi o khoa source_metrics_mtime) chu khong dung ban doc tu dau '
        'notebook. Khoa mang hau to _pp la gia tri quy doi sang diem phan tram; moi khoa con lai o '
        'thang [0,1]. Toan bo chay tren GPU RTX 4060 Laptop, PyTorch 2.13.0+cu126.'
    ),
}

out_path = os.path.join(REP_DIR, 'metrics_ablation.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

# --- Kiem tra lai tep vua ghi: doc lai va doi chieu cac khoa bat buoc ---
with open(out_path, encoding='utf-8') as f:
    chk = json.load(f)
required = {
    'factorial': ['combinations', 'main_effects', 'interactions', 'n_train', 'epochs'],
    'learning_curve': ['n_values', 'mean', 'std', 'seeds'],
    'gap_decomposition': ['a_numpy', 'a_small', 'a_full', 'n_small', 'n_full',
                          'data_component', 'residual_component', 'total_gap'],
}
for sec, keys in required.items():
    missing = [k for k in keys if k not in chk[sec]]
    assert not missing, f'Thieu khoa {missing} trong {sec}'
assert 'notes' in chk and len(chk['notes']) > 100
assert len(chk['factorial']['combinations']) == 8
assert len(chk['learning_curve']['n_values']) == len(chk['learning_curve']['mean']) \
       == len(chk['learning_curve']['std'])
for c in chk['factorial']['combinations']:
    assert {'padding', 'he_init', 'lr_decay', 'accuracy'}.issubset(c)

print('Đã ghi', out_path, '|', f'{os.path.getsize(out_path)/1024:.1f} KB')
print('Kiểm tra lược đồ: đủ mọi khóa bắt buộc, 8 tổ hợp giai thừa, '
      f'{len(chk["learning_curve"]["n_values"])} điểm đường cong.')
print()
for f_ in ('fig_ablation_factorial.png', 'fig_learning_curve.png', 'fig_gap_decomposition.png'):
    p = os.path.join(FIG_DIR, f_)
    if os.path.exists(p):
        print(f'  {f_:<32} có    {os.path.getsize(p)/1024:6.0f} KB')
    else:
        print(f'  {f_:<32} THIẾU')

Đã ghi ../reports\metrics_ablation.json | 13.2 KB
Kiểm tra lược đồ: đủ mọi khóa bắt buộc, 8 tổ hợp giai thừa, 8 điểm đường cong.

  fig_ablation_factorial.png       có       124 KB
  fig_learning_curve.png           có       138 KB
  fig_gap_decomposition.png        có       122 KB


<!--INTERP:conclusion-->

Kết luận đang chờ số liệu thực thi.